# GazeDecoder V3 — Baseline Comparison

Compares the best ChronosX variant (identified by `ablation.ipynb`) against
12 baseline models using LOSO cross-validation.

**Prerequisites:** Run `ablation.ipynb` first to generate
`archive/ablation/best_variant.json`.

**Sections**
1. Environment Setup
2. Package Installation
3. Dataset
4. Best ChronosX from Ablation
5. LOSO — All Baselines
6. Results Visualization
7. Statistical Tests
8. Scott-Knott ESD Ranking

## §1  Environment Setup

In [ ]:
import sys, os, json, pathlib

# ── Mount Google Drive (Colab) ───────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    COLAB = True
except ImportError:
    COLAB = False
    print("[Local mode] Drive not mounted.")

# ── Locate code directory ────────────────────────────────────────────────────
if COLAB:
    CODE_DIR = pathlib.Path("/content/drive/MyDrive/EyeSeq/code/GazeDecoderV3")
else:
    CODE_DIR = pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path(".").resolve()

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print(f"CODE_DIR = {CODE_DIR}")
print(f"COLAB    = {COLAB}")

## §2  Package Installation

In [ ]:
if COLAB:
    %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    %pip install -q scikit-learn xgboost lightgbm sentence-transformers tqdm seaborn scipy
    print("✅ Packages installed.")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from shared.config   import (SEED, GAZE_DIR, ECONTEXT_PATH,
                              BASE_DIR, ABL_DIR, DS_CACHE_DIR,
                              WINDOW_SIZE, STRIDE, HIT_THRESHOLD,
                              FEAT_DIM, set_global_seed)
from shared.dataset  import build_dataset, get_loso_splits
from shared.models   import CHRONOSX_VARIANTS, BASELINE_MODELS
from shared.training import run_loso, run_all_models
from shared.viz      import (results_to_df, plot_f1_leaderboard, plot_prf_grouped,
                              plot_mean_confmat, plot_training_curves,
                              significance_table, scott_knott_esd, plot_scott_knott)

set_global_seed()
print(f"SEED={SEED} | torch={torch.__version__} | device={'cuda' if torch.cuda.is_available() else 'cpu'}")

## §3  Dataset

Reuses the same cache as `ablation.ipynb` — no re-computation needed.

In [ ]:
ds = build_dataset(
    gaze_dir       = GAZE_DIR,
    econtext_path  = ECONTEXT_PATH,
    cache_dir      = DS_CACHE_DIR,
    window_size    = WINDOW_SIZE,
    stride         = STRIDE,
    hit_threshold  = HIT_THRESHOLD,
)
splits = get_loso_splits(ds)
print(f"Dataset: {len(ds)} windows | {len(splits)} subjects (folds)")

## §4  Best ChronosX from Ablation

Reads `archive/ablation/best_variant.json` written by `ablation.ipynb`
and runs (or resumes) that single variant's LOSO experiment under
`archive/baselines/<variant>/` so results sit alongside the baselines.

In [ ]:
best_json = ABL_DIR / "best_variant.json"
if not best_json.exists():
    raise FileNotFoundError(
        f"❌ '{best_json}' not found — please run ablation.ipynb first.")

with open(best_json) as f:
    _bv = json.load(f)

BEST_CHRONOSX = _bv["best_variant"]
print(f"Best ChronosX variant: {BEST_CHRONOSX}")
print(f"  F1-issue (ablation):  {_bv['f1_issue']:.4f}")
print(f"  F1-macro (ablation):  {_bv['f1_macro']:.4f}")

In [ ]:
# Run best ChronosX in the baselines archive (separate from ablation folds)
proposed_registry = {BEST_CHRONOSX: CHRONOSX_VARIANTS[BEST_CHRONOSX]}
proposed_result = run_all_models(
    registry     = proposed_registry,
    ds           = ds,
    archive_root = BASE_DIR,
    names        = None,
    verbose      = True,
)
print(f"\n✅ {BEST_CHRONOSX} LOSO complete.")

## §5  LOSO — All 12 Baselines

All 12 baselines run under identical LOSO conditions.
Results are cached to `archive/baselines/<model>/fold_NN.json`.
Completed folds are skipped on re-run.

| Category | Models |
|---|---|
| ML | XGBoost, RandomForest, LightGBM |
| DL-RNN | BiLSTM |
| DL-CNN | 1D-CNN |
| DL-Attention | TransformerEnc, DLinear, PatchTST, iTransformer, TimesNet |
| DL-SSM | Mamba |
| DL-LLM | TimesBERT |

In [ ]:
baseline_results = run_all_models(
    registry     = BASELINE_MODELS,
    ds           = ds,
    archive_root = BASE_DIR,
    names        = None,   # run all 12
    verbose      = True,
)
print("\n✅ All baselines complete.")

## §6  Results Visualization

In [ ]:
# Merge best ChronosX + all baselines into one results dict
all_results = {**proposed_result, **baseline_results}

all_df = results_to_df(all_results)
print("── Full comparison summary ───────────────────────────────────────────────")
display(all_df)

In [ ]:
plot_f1_leaderboard(
    all_df,
    title     = f"F1-Issue (LOSO mean) — {BEST_CHRONOSX} vs 12 Baselines",
    save_path = str(BASE_DIR / "fig_f1_leaderboard.png"),
)
plot_prf_grouped(
    all_df,
    top_k     = min(10, len(all_df)),
    title     = f"Precision / Recall / F1-Issue — Top-10 models",
    save_path = str(BASE_DIR / "fig_prf_top10.png"),
)

In [ ]:
# Confusion matrices — proposed + top DL baselines
TOP_DL = [BEST_CHRONOSX] + [
    m for m in all_df["model"] if m != BEST_CHRONOSX
][:5]
for m in TOP_DL:
    plot_mean_confmat(all_results, m)

# Training curves — DL models only
for name, rep in all_results.items():
    if rep.get("histories"):
        plot_training_curves(
            rep["histories"],
            title     = f"{name} — fold-averaged training curves",
            save_path = str(BASE_DIR / f"fig_curves_{name}.png"),
        )

## §7  Statistical Tests

Pairwise significance: best ChronosX vs each baseline.
- Shapiro-Wilk normality on fold differences → paired-t or Wilcoxon
- Bootstrap 95% CI on mean Δ F1-issue
- Effect size: Cohen's d (parametric) or Cliff's δ (non-parametric)

In [ ]:
sig_df = significance_table(
    all_results,
    proposed = BEST_CHRONOSX,
    metric   = "f1_issue",
)
print(f"Significance: {BEST_CHRONOSX} vs baselines (sorted by p-value)")
display(sig_df)

sig_out = BASE_DIR / "significance_table.csv"
sig_df.to_csv(sig_out, index=False)
print(f"\n✅ Saved: {sig_out}")

## §8  Scott-Knott ESD Ranking

Groups models into statistically homogeneous tiers (Hedges' g < 0.2 = negligible difference).
Tier 1 is the best group.

In [ ]:
sk_df = scott_knott_esd(all_results, metric="f1_issue")
print("Scott-Knott ESD tiers:")
display(sk_df)

sk_out = BASE_DIR / "scott_knott.csv"
sk_df.to_csv(sk_out, index=False)
print(f"\n✅ Saved: {sk_out}")

plot_scott_knott(
    sk_df,
    proposed  = BEST_CHRONOSX,
    metric    = "f1_issue",
    title     = f"Scott-Knott ESD — {BEST_CHRONOSX} vs 12 Baselines",
    save_path = str(BASE_DIR / "fig_scott_knott.png"),
)